# Y-Scrambling Validation — RF-1 (Avalon + 5 Descriptors)

Standalone notebook. It rebuilds the features exactly as in the main notebook (same code, copied verbatim), then runs a **proper y-scrambling test**:

1. Shuffle the **training** labels (`y_dev`)
2. **Retrain** the RF with the same hyperparameters (nT=150, Td=30)
3. Evaluate on the **real** test labels (`y_test`)
4. Repeat N times and compare to the real model

If the scrambled models sit near chance (accuracy ≈ 0.5, AUROC ≈ 0.5) while the real model is far above, the performance reflects real structure–activity signal rather than chance correlation.

**Before running:** upload `train.csv`, `validation.csv`, `test.csv` to the Colab working directory (or edit paths in Section 2).

## 0. Install dependencies

In [ ]:
import subprocess, sys
for pkg in ["rdkit", "pandas", "numpy", "scikit-learn", "matplotlib", "seaborn", "tqdm"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

# Install openchemlib-JS via npm (used for descriptor calculation)
subprocess.run(['npm', 'install', 'openchemlib', '--prefix', '/tmp/ocl'],
               check=True, capture_output=True)
print("All packages installed (including openchemlib-JS).")


## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import subprocess, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.notebook import tqdm
import pickle, os

# RDKit — only for Avalon fingerprints and SMILES validation
from rdkit import Chem, DataStructs

# Avalon
try:
    from rdkit.Avalon import pyAvalonTools
    AVALON_AVAILABLE = True
    print("Avalon fingerprints available")
except ImportError:
    AVALON_AVAILABLE = False
    print("WARNING: Avalon not available. Falling back to RDKit topological FP.")

# Scikit-learn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, matthews_corrcoef,
    cohen_kappa_score, confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import cdist

print("All imports successful")


## 2. Configuration

In [ ]:
import os

# ============================================================
#  EDIT THESE to match your files
# ============================================================

TRAIN_CSV  = "train.csv"        # 60% — used for RF training
VAL_CSV    = "validation.csv"   # 20% — used for hyperparameter optimisation
TEST_CSV   = "test.csv"         # 20% — held-out external test set

SMILES_COL = "Molecule (RDKit Mol) (Canonical)"  # Column with SMILES strings
LABEL_COL  = "Category"                           # Activity: 1 = active, 0 = inactive

# NOTE: BaN is now calculated automatically via OpenChemLib-JS.
# No BAN_COL or DataWarrior CSV needed.

AVALON_NBITS = 1024   # Paper used 1024

# ============================================================
print(f"Train CSV  : {TRAIN_CSV}")
print(f"Val CSV    : {VAL_CSV}")
print(f"Test CSV   : {TEST_CSV}")
print(f"SMILES col : {SMILES_COL}")
print(f"Label col  : {LABEL_COL}")
print(f"Avalon bits: {AVALON_NBITS}")
print(f"Descriptor backend: OpenChemLib-JS (Node.js)")


## 3. Load data

In [ ]:
def load_csv(path, label):
    df = pd.read_csv(path)
    print(f"{label:20s}: {len(df):,} rows  |  columns: {list(df.columns)}")
    return df

train_df = load_csv(TRAIN_CSV, "Training set")
val_df   = load_csv(VAL_CSV,   "Validation set")
test_df  = load_csv(TEST_CSV,  "External test set")

# Validate required columns
for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    for col in [SMILES_COL, LABEL_COL]:
        assert col in df.columns, f"Column '{col}' not found in {name} CSV!"

# Class distribution
print("\nClass distribution:")
for name, df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    vc = df[LABEL_COL].value_counts().sort_index()
    print(f"  {name:8s}: Active={vc.get(1,0):,}  Inactive={vc.get(0,0):,}  Total={len(df):,}")


## 4. Feature building (identical to main notebook)

In [ ]:
# ── OpenChemLib-JS descriptor calculation ──────────────────────────────────
# basicNitrogenCount is a line-by-line port of:
#   AtomFunctionAnalyzer.isBasicNitrogen(StereoMolecule mol, int atom)
# from the OpenChemLib Java source (Actelion Pharmaceuticals Ltd., BSD-3 licence).

_OCL_LIB = '/tmp/ocl/node_modules/openchemlib/dist/openchemlib.js'

_OCL_JS_SCRIPT = r"""
import OCL from '""" + _OCL_LIB + r"""';
const { Molecule, MoleculeProperties } = OCL;

function getFakeOxoCount(mol, atom) {
  let n = 0;
  for (let i = 0; i < mol.getConnAtoms(atom); i++) {
    const no = mol.getAtomicNo(mol.getConnAtom(atom, i));
    if (mol.getConnBondOrder(atom, i) === 2 && (no===7||no===8||no===16)) n++;
  }
  return n;
}

function isStabilized(mol, atom, twice) {
  let found = false;
  for (let i = 0; i < mol.getConnAtoms(atom); i++) {
    const bond = mol.getConnBond(atom, i);
    if (!mol.isAromaticBond(bond) && mol.getConnBondOrder(atom, i) === 1) {
      const conn = mol.getConnAtom(atom, i);
      if (!mol.isAromaticAtom(conn) &&
          ((mol.getAtomicNo(conn)===6 && getFakeOxoCount(mol,conn)===1) ||
           (mol.getAtomicNo(conn)===16 && getFakeOxoCount(mol,conn)===2))) {
        if (found || !twice) return true;
        found = true;
      }
    }
  }
  return false;
}

function isVinylogFakeOxo(mol, atom) {
  for (let i = 0; i < mol.getConnAtoms(atom); i++) {
    if (mol.getConnBondOrder(atom, i) !== 1) {
      const conn = mol.getConnAtom(atom, i);
      for (let j = 0; j < mol.getConnAtoms(conn); j++)
        if (mol.getConnBondOrder(conn,j)===1 && getFakeOxoCount(mol,mol.getConnAtom(conn,j))!==0)
          return true;
    }
  }
  return false;
}

function isBasicNitrogen(mol, atom) {
  if (mol.getAtomicNo(atom) !== 7) return false;
  if (mol.getAtomCharge(atom) !== 0) return false;
  if (mol.getConnAtoms(atom) + mol.getAtomPi(atom) > 3) return false;
  if (mol.isAromaticAtom(atom)) {
    if (mol.getAtomPi(atom) !== 1) return false;
    if (mol.getAtomRingCount(atom, 7) !== 1) return false;
    const rc = mol.getRingSet();
    for (let r = 0; r < rc.getSize(); r++) {
      if (!rc.isAtomMember(r, atom)) continue;
      const sz = rc.getRingSize(r);
      if (sz !== 5 && sz !== 6) { break; }
      const ring = rc.getRingAtoms(r);
      const R = Array.from({length:sz}, (_,i)=>ring[i]);
      let ni = R.indexOf(atom);
      let opi, mi;
      if (sz === 5) {
        opi = [R[ni-1<0?ni+4:ni-1], R[ni-4<0?ni+1:ni-4]];
        mi  = [R[ni-2<0?ni+3:ni-2], R[ni-3<0?ni+2:ni-3]];
      } else {
        opi = [R[ni-1<0?ni+5:ni-1], R[ni-3<0?ni+3:ni-3], R[ni-5<0?ni+1:ni-5]];
        mi  = [R[ni-2<0?ni+4:ni-2], R[ni-4<0?ni+2:ni-4]];
      }
      let ec = 0;
      for (let i=0; i<R.length; i++)
        if (atom!==R[i] && mol.getAtomicNo(R[i])===7 && mol.getAtomPi(R[i])===1) ec--;
      for (let i=0; i<opi.length; i++) {
        let exoAtom=-1, exoBond=-1;
        for (let j=0; j<mol.getConnAtoms(opi[i]); j++) {
          if (!mol.isAromaticBond(mol.getConnBond(opi[i],j))) {
            exoAtom=mol.getConnAtom(opi[i],j); exoBond=mol.getConnBond(opi[i],j); break;
          }
        }
        if (exoAtom !== -1) {
          if (mol.getAtomicNo(exoAtom)===7 && mol.getAtomPi(exoAtom)===0
              && (mol.getConnAtoms(exoAtom)+mol.getAtomPi(exoAtom)<=3)
              && !isStabilized(mol,exoAtom,false)) { ec++; continue; }
          if (mol.getAtomicNo(exoAtom)===8 && mol.getConnAtoms(exoAtom)===1) { ec+=2; continue; }
          if (mol.isAromaticBond(exoBond)) {
            for (let s=0; s<rc.getSize(); s++) {
              if (rc.isAromatic(s) && rc.isAtomMember(s,exoAtom)) {
                const ra=rc.getRingAtoms(s);
                for (let j=0; j<rc.getRingSize(s); j++)
                  if (mol.getAtomicNo(ra[j])===7 && mol.getAtomPi(ra[j])===1) { ec--; break; }
                break;
              }
            }
          }
        }
      }
      for (let i=0; i<mi.length; i++) {
        let exoAtom=-1;
        for (let j=0; j<mol.getConnAtoms(mi[i]); j++)
          if (!mol.isAromaticBond(mol.getConnBond(mi[i],j))) exoAtom=mol.getConnAtom(mi[i],j);
        if (mol.getAtomicNo(mi[i])===6) {
          if (exoAtom!==-1 && getFakeOxoCount(mol,exoAtom)!==0) ec--;
        } else if (mol.getAtomicNo(mi[i])===7) {
          if (mol.getAtomPi(mi[i])===0 &&
              (exoAtom===-1 || (!mol.isAromaticAtom(exoAtom) && getFakeOxoCount(mol,exoAtom)===0)))
            ec++;
        }
      }
      return ec > 0;
    }
    return false;
  }
  if (mol.getAtomPi(atom) > 1) return false;
  if (mol.getAtomPi(atom) === 1) {
    let imineC=-1, sc=0;
    for (let i=0; i<mol.getConnAtoms(atom); i++) {
      const conn=mol.getConnAtom(atom,i);
      if (mol.getConnBondOrder(atom,i)===2) {
        if (mol.getAtomicNo(conn)!==6) return false;
        imineC=conn; continue;
      }
      if (mol.getAtomicNo(conn)===8) return false;
      if (mol.getAtomicNo(conn)===7) { sc--; if (isStabilized(mol,conn,false)) sc--; continue; }
      if (mol.isAromaticAtom(conn)) sc--;
    }
    if (imineC===-1) return false;
    let arNC=0;
    for (let i=0; i<mol.getConnAtoms(imineC); i++) {
      if (mol.getConnBondOrder(imineC,i)===1) {
        const conn=mol.getConnAtom(imineC,i);
        if (getFakeOxoCount(mol,conn)!==0) return false;
        if (mol.isAromaticAtom(conn)) arNC++;
        if (mol.getAtomicNo(conn)===7 && !isStabilized(mol,conn,true)) sc++;
        if (mol.getAtomicNo(conn)===8||mol.getAtomicNo(conn)===16) sc--;
      }
    }
    if (arNC===2) sc--;
    return sc >= 0;
  }
  for (let i=0; i<mol.getConnAtoms(atom); i++) {
    const conn=mol.getConnAtom(atom,i);
    if (mol.isAromaticAtom(conn)) return false;
    if (mol.getAtomicNo(conn)!==6) return false;
    if (getFakeOxoCount(mol,conn)!==0) return false;
    if (mol.getAtomPi(conn)!==0 && isVinylogFakeOxo(mol,conn)) return false;
  }
  return true;
}

function countBasicNitrogens(mol) {
  let n=0;
  for (let i=0; i<mol.getAllAtoms(); i++) if (isBasicNitrogen(mol,i)) n++;
  return n;
}

const smilesList = JSON.parse(process.argv[2]);
const results = [];
for (const smi of smilesList) {
  try {
    const mol = Molecule.fromSmiles(smi);
    mol.ensureHelperArrays(255);
    const props = new MoleculeProperties(mol);
    const formula = mol.getMolecularFormula();
    results.push({
      smi,
      mw:    parseFloat(formula.relativeWeight.toFixed(3)),
      tpsa:  parseFloat(props.polarSurfaceArea.toFixed(3)),
      clogp: parseFloat(props.logP.toFixed(3)),
      arom:  mol.getAromaticRingCount(),
      ban:   countBasicNitrogens(mol),
    });
  } catch(e) {
    results.push({smi, error: e.message});
  }
}
process.stdout.write(JSON.stringify(results));
"""


def run_ocl_descriptors(smiles_list: list, batch_size: int = 500) -> dict:
    """
    Calculate [MW, TPSA, cLogP, AromaticRings, BaN] for a list of SMILES
    using OpenChemLib-JS. Batches calls to avoid OS arg-length limits.
    Returns a dict: smiles -> np.array([5], float32).
    """
    if not smiles_list:
        return {}

    with open('/tmp/_ocl_desc.mjs', 'w') as f:
        f.write(_OCL_JS_SCRIPT)

    out = {}
    # Split into chunks to stay under OS argument length limit
    for start in range(0, len(smiles_list), batch_size):
        chunk = smiles_list[start : start + batch_size]
        result = subprocess.run(
            ['node', '/tmp/_ocl_desc.mjs', json.dumps(chunk)],
            capture_output=True, text=True, timeout=300
        )
        if result.returncode != 0:
            print(f'[OCL] Node.js error on batch {start//batch_size}: {result.stderr[:400]}')
            continue
        rows = json.loads(result.stdout)
        for r in rows:
            if 'error' in r:
                continue
            out[r['smi']] = np.array(
                [r['mw'], r['tpsa'], r['clogp'], r['arom'], r['ban']],
                dtype=np.float32
            )

    return out


def calc_avalon_fp(mol, n_bits=1024):
    """Avalon fingerprint as float32 numpy array (falls back to RDKit FP)."""
    if AVALON_AVAILABLE:
        from rdkit.Avalon import pyAvalonTools
        fp = pyAvalonTools.GetAvalonFP(mol, nBits=n_bits)
    else:
        fp = Chem.RDKFingerprint(mol, fpSize=n_bits)
    arr = np.zeros(fp.GetNumBits(), dtype=np.float32)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


DESC_NAMES = ['MW', 'TPSA', 'cLogP', 'AromaticRings', 'BaN']
print(f"Feature vector: [{AVALON_NBITS} Avalon bits | {' | '.join(DESC_NAMES)}] = {AVALON_NBITS + 5} total features")
print("Descriptor backend: OpenChemLib-JS (Node.js subprocess)")
print("  BaN: pKa-based port of AtomFunctionAnalyzer.isBasicNitrogen()")


In [ ]:
def build_feature_matrix(df, smiles_col, label_col,
                          n_bits=1024, scaler=None, fit_scaler=False, set_name="Set"):
    """
    Build X = [Avalon FP | standardised 5 descriptors], y = labels.
    Descriptors (MW, TPSA, cLogP, AromaticRings, BaN) are computed via OpenChemLib-JS.

    Parameters
    ----------
    fit_scaler : True only for training set — fits and returns the scaler.
                 For val/test, pass the fitted scaler and set fit_scaler=False.

    Returns
    -------
    X, y, valid_df, [scaler if fit_scaler else None]
    """
    print(f"\nBuilding features for {set_name} ({len(df):,} molecules)...")

    # --- Step 1: validate SMILES and collect canonical forms ---
    valid_idx, valid_smiles = [], []
    for original_idx, row in df.iterrows():
        smi = str(row[smiles_col])
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            valid_idx.append(original_idx)
            valid_smiles.append(smi)  # keep original SMILES for OCL (matches df row)

    valid_df = df.loc[valid_idx].reset_index(drop=True)
    print(f"  Dropped: {len(df) - len(valid_idx)} invalid SMILES")

    # --- Step 2: batch descriptor calculation via OpenChemLib-JS ---
    print(f"  [OCL] Calculating descriptors for {len(valid_smiles)} molecules...")
    desc_map = run_ocl_descriptors(valid_smiles)

    # --- Step 3: build fingerprint + descriptor matrices ---
    fp_list, desc_list, keep_idx = [], [], []
    for i, (smi, orig_idx) in enumerate(zip(valid_smiles, valid_idx)):
        if smi not in desc_map:
            continue  # OCL failed on this one
        mol = Chem.MolFromSmiles(smi)
        try:
            fp = calc_avalon_fp(mol, n_bits=n_bits)
        except Exception as e:
            print(f"  FP error for {smi}: {e}")
            continue
        fp_list.append(fp)
        desc_list.append(desc_map[smi])
        keep_idx.append(i)

    valid_df = valid_df.iloc[keep_idx].reset_index(drop=True)
    fp_mat   = np.array(fp_list,   dtype=np.float32)  # (n, 1024)
    desc_mat = np.array(desc_list, dtype=np.float32)  # (n, 5)

    # --- Step 4: scale descriptors ---
    if fit_scaler:
        scaler = StandardScaler()
        desc_scaled = scaler.fit_transform(desc_mat)
        print(f"  Scaler fitted on {set_name}")
    elif scaler is not None:
        desc_scaled = scaler.transform(desc_mat)
    else:
        desc_scaled = desc_mat

    X = np.hstack([fp_mat, desc_scaled])  # (n, 1029)

    # --- Step 5: encode labels ---
    label_mapping = {'Active': 1, 'Inactive': 0}
    y = valid_df[label_col].map(label_mapping).values.astype(int)

    print(f"  Shape: {X.shape}  |  Active: {y.sum():,}  Inactive: {(y==0).sum():,}")
    print(f"  Descriptor backend: OpenChemLib-JS")

    return (X, y, valid_df, scaler) if fit_scaler else (X, y, valid_df, None)


# Build training set and fit scaler
X_train, y_train, train_valid_df, scaler = build_feature_matrix(
    train_df, SMILES_COL, LABEL_COL,
    n_bits=AVALON_NBITS, fit_scaler=True, set_name="Training"
)

# Validation set — apply training scaler
X_val, y_val, val_valid_df, _ = build_feature_matrix(
    val_df, SMILES_COL, LABEL_COL,
    n_bits=AVALON_NBITS, scaler=scaler, set_name="Validation"
)

# Test set — apply training scaler
X_test, y_test, test_valid_df, _ = build_feature_matrix(
    test_df, SMILES_COL, LABEL_COL,
    n_bits=AVALON_NBITS, scaler=scaler, set_name="Test"
)

# Combined dev set (train + val) for final model training and CV
X_dev = np.vstack([X_train, X_val])
y_dev = np.concatenate([y_train, y_val])

# Avalon-only slices for APD
X_train_fp = X_train[:, :AVALON_NBITS]
X_test_fp  = X_test[:,  :AVALON_NBITS]

print(f"\nSummary:")
print(f"  Train : {X_train.shape}")
print(f"  Val   : {X_val.shape}")
print(f"  Test  : {X_test.shape}")
print(f"  Dev   : {X_dev.shape}")
print(f"  Features: {AVALON_NBITS} Avalon + 5 descriptors ({', '.join(DESC_NAMES)}) = {X_train.shape[1]}")


## 5. Metrics helpers

In [ ]:
def compute_metrics(y_true, y_pred, y_prob=None):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    m = {
        'Accuracy':    accuracy_score(y_true, y_pred),
        'Precision':   precision_score(y_true, y_pred, zero_division=0),
        'Sensitivity': recall_score(y_true, y_pred, zero_division=0),
        'Specificity': tn / (tn + fp) if (tn + fp) > 0 else 0.0,
        'F1':          f1_score(y_true, y_pred, zero_division=0),
        'Kappa':       cohen_kappa_score(y_true, y_pred),
        'MCC':         matthews_corrcoef(y_true, y_pred),
    }
    if y_prob is not None:
        m['AUROC'] = roc_auc_score(y_true, y_prob)
    m.update({'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn)})
    return m


def print_metrics(m, title="Metrics"):
    print(f"\n{'='*52}\n  {title}\n{'='*52}")
    for k, v in m.items():
        print(f"  {k:15s}: {v:.4f}" if isinstance(v, float) else f"  {k:15s}: {v}")

print("Metrics functions ready")

### Sanity check on labels
The main notebook's class-distribution printout (`value_counts().get(1)`) and the encoded `y` (via the `'Active'/'Inactive'` map) gave different counts for Active vs Inactive. Confirm the encoding here before trusting anything downstream.

In [ ]:
print("Raw label values in train:", train_df[LABEL_COL].unique()[:5])
print("Encoded y_dev  -> 1s:", int(y_dev.sum()), " 0s:", int((y_dev == 0).sum()))
print("Encoded y_test -> 1s:", int(y_test.sum()), " 0s:", int((y_test == 0).sum()))
assert set(np.unique(y_dev)) == {0, 1} and set(np.unique(y_test)) == {0, 1}
print(f"Test positive rate: {y_test.mean():.3f}  (chance accuracy for a random classifier ≈ 0.5 only if classes ~balanced)")

## 6. Real model (baseline)
Hyperparameters taken from the main notebook's grid search: **nT=150, Td=30**.

In [ ]:
BEST_NT, BEST_TD = 150, 30

rf_real = RandomForestClassifier(n_estimators=BEST_NT, max_depth=BEST_TD,
                                 n_jobs=-1, random_state=42)
rf_real.fit(X_dev, y_dev)

real_acc = accuracy_score(y_test, rf_real.predict(X_test))
real_auc = roc_auc_score(y_test, rf_real.predict_proba(X_test)[:, 1])
print(f"Real model  ->  Accuracy: {real_acc:.4f}   AUROC: {real_auc:.4f}")

## 7. Y-scrambling
Training labels are permuted, the model is refit, and it is scored against the **unshuffled** test labels.

Runtime: 100 refits of a 150-tree forest. Set `N_SCRAMBLES = 20` for a quick check; 100 for the final result.

In [ ]:
N_SCRAMBLES = 100

rng = np.random.RandomState(42)
scr_acc, scr_auc = [], []

for i in tqdm(range(N_SCRAMBLES), desc="Y-scrambling"):
    y_shuf = rng.permutation(y_dev)                       # scramble TRAINING labels only
    rf_s = RandomForestClassifier(n_estimators=BEST_NT, max_depth=BEST_TD,
                                  n_jobs=-1, random_state=i)
    rf_s.fit(X_dev, y_shuf)                               # retrain from scratch
    scr_acc.append(accuracy_score(y_test, rf_s.predict(X_test)))              # real test labels
    scr_auc.append(roc_auc_score(y_test, rf_s.predict_proba(X_test)[:, 1]))

scr_acc, scr_auc = np.array(scr_acc), np.array(scr_auc)

# Empirical one-sided p-value (with +1 correction)
p_acc = (1 + np.sum(scr_acc >= real_acc)) / (N_SCRAMBLES + 1)
p_auc = (1 + np.sum(scr_auc >= real_auc)) / (N_SCRAMBLES + 1)

print(f"\nY-scrambling results ({N_SCRAMBLES} runs)")
print(f"  Accuracy : real = {real_acc:.4f} | scrambled = {scr_acc.mean():.4f} ± {scr_acc.std():.4f} "
      f"(max {scr_acc.max():.4f}) | p ≤ {p_acc:.4f}")
print(f"  AUROC    : real = {real_auc:.4f} | scrambled = {scr_auc.mean():.4f} ± {scr_auc.std():.4f} "
      f"(max {scr_auc.max():.4f}) | p ≤ {p_auc:.4f}")

## 8. Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, scr, real, name in [(axes[0], scr_acc, real_acc, 'Accuracy'),
                            (axes[1], scr_auc, real_auc, 'AUROC')]:
    ax.hist(scr, bins=25, color='salmon', alpha=0.75, edgecolor='k',
            label=f'Y-scrambled (mean={scr.mean():.3f})')
    ax.axvline(real, color='steelblue', lw=2.5, ls='--', label=f'Real model ({real:.3f})')
    ax.axvline(0.5, color='gray', lw=1.5, ls=':', label='Chance (0.500)')
    ax.set_xlabel(name); ax.set_ylabel('Count')
    ax.set_xlim(0.4, 1.0)
    ax.set_title(f'Y-Scrambling — {name}', fontweight='bold')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('y_scrambling.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
res = pd.DataFrame({'scrambled_accuracy': scr_acc, 'scrambled_auroc': scr_auc})
res.to_csv('y_scrambling_results.csv', index=False)
print("Saved y_scrambling_results.csv and y_scrambling.png")